# Araba Yıkama Tahmin Projesi - Makine Öğrenmesi

**İskenderun 20 yıllık hava durumu** ile **regresyon** kullanarak yarınki yağış miktarını tahmin ediyoruz.
Karar: *Bugün arabamı yıkamalı mıyım?* → Yarın yağmur varsa yıkama.

---

## 1. Veri Çekme

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..")
sys.path.insert(0, str(ROOT / "src"))

from data_fetch import fetch_all

# 20 yıllık İskenderun hava verisi (ilk çalıştırmada ~1-2 dk, data/raw/ altına kaydeder)
fetch_all(20)

## 2. Veri Ön İşleme ve Özellik Mühendisliği

In [ ]:
from preprocess import build_ml_data, get_feature_columns

df = build_ml_data()
features = get_feature_columns()

print(f"Satır: {len(df)}, Özellik: {len(features)}")
df[features + ["yagmur_yarin"]].head(10)

## 3. Regresyon Modeli Eğitimi

In [ ]:
from train import load_data, train_model, save_model

X, y, df = load_data()
model, scaler = train_model(X, y)
save_model(model, scaler)

## 4. Örnek Tahmin ve Öneri

In [ ]:
import pickle
from preprocess import get_recommendation
from train import load_data

model_path = ROOT / "models" / "model.pkl"
scaler_path = ROOT / "models" / "scaler.pkl"

if model_path.exists():
    # X yoksa (Cell 6 atlanmışsa) veriyi yükle
    try:
        _ = X
    except NameError:
        X, y, df = load_data()
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)
    
    # Son günün verisiyle tahmin
    X_son = X.iloc[[-1]]
    X_son_s = scaler.transform(X_son)
    tahmin = model.predict(X_son_s)[0]
    
    print(f"Yarın tahmini yağış: {tahmin:.2f} mm")
    print(get_recommendation(tahmin))